# 🔬 Model Pipeline: Google Pegasus-PubMed (Seq2Seq Baseline)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `google/pegasus-pubmed`
**Architecture:** SEQ2SEQ (LoRA/FP16)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json
import pandas as pd
import numpy as np
import torch

BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Drive ready!")

In [ ]:

# CELL 3 — DOWNLOAD OFFICIAL DATASETS AUTOMATICALLY
!pip -q install gdown

import os
import gdown
import pandas as pd

DATASET_DIR = "/content/drive/MyDrive/JU_Benchmark_Project/Dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

files = {
    "train": (
        "1oydpEnDZ8Chu7dQpT5RGnnRTMvhfq-8Q",
        "MixSub-SciHigh_train_FIRE.csv"
    ),
    "val": (
        "1-A0tc0sFJGgwWEOywM9Ddyxi6G4K_Xhj",
        "MixSub-SciHigh_val_FIRE.csv"
    )
}

for name, (file_id, filename) in files.items():
    path = os.path.join(DATASET_DIR, filename)

    if not os.path.exists(path):
        print("Downloading", filename)
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, path, quiet=False)

    if not os.path.exists(path) or os.path.getsize(path) == 0:
        raise RuntimeError(f"Download failed: {filename}")

    print("READY:", filename)

TRAIN_CSV = os.path.join(DATASET_DIR, files["train"][1])
VAL_CSV = os.path.join(DATASET_DIR, files["val"][1])

val_df = pd.read_csv(VAL_CSV)
val_df = val_df.dropna(subset=["Abstract", "Highlights"]).reset_index(drop=True)

# Masked test file is not linked in the official page's visible download links.
# Use validation rows only as a temporary inference smoke test.
TEST_CSV = None
test_df = val_df[["Abstract"]].copy()

print("\nDATASETS LOADED SUCCESSFULLY")
print("Training:", len(pd.read_csv(TRAIN_CSV)))
print("Validation:", len(val_df))
print("Temporary inference samples:", len(test_df))
print("\nWARNING: Temporary inference uses validation data, NOT the official masked test set.")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "google/pegasus-pubmed"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading Pegasus-PubMed...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)

model.to(device)
model.eval()

print("Pegasus-PubMed loaded successfully!")

In [ ]:
import re
from collections import Counter
from rouge_score import rouge_scorer

# Keep these limits for a quick first run.
# Increase them after confirming everything works.
VAL_SAMPLES = 10
TEST_SAMPLES = 20
BATCH_SIZE = 4

val_run = val_df.head(VAL_SAMPLES).copy()
test_run = test_df.head(TEST_SAMPLES).copy()

def generate_predictions(texts):
    predictions = []
    latencies = []
    token_counts = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start+BATCH_SIZE]
        prompts = ["summarize: " + str(x) for x in batch]

        inputs = tokenizer(
            prompts,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        if device == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=1,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        if device == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time
        batch_latency = elapsed / len(batch)

        decoded = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        predictions.extend([x.strip() for x in decoded])
        latencies.extend([batch_latency] * len(batch))
        token_counts.extend([
            int((row != tokenizer.pad_token_id).sum().item())
            for row in outputs
        ])

        print(
            f"Processed {min(start+len(batch), len(texts))}"
            f"/{len(texts)}"
        )

    return predictions, latencies, token_counts


# 1. Validation predictions
print("\nGenerating validation predictions...")

val_predictions, val_latencies, val_tokens = generate_predictions(
    val_run["Abstract"].astype(str).tolist()
)

# 2. Calculate ROUGE-L
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = [
    scorer.score(str(ref), str(pred))["rougeL"].fmeasure
    for ref, pred in zip(val_run["Highlights"], val_predictions)
]

# 3. Calculate source grounding and target F1
def calculate_metrics(pred, source, target):
    tokenize = lambda x: re.findall(
        r"\b[a-zA-Z0-9]+\b", str(x).lower()
    )

    p = Counter(tokenize(pred))
    s = Counter(tokenize(source))
    t = Counter(tokenize(target))

    total = sum(p.values())

    source_overlap = sum((p & s).values())
    target_overlap = sum((p & t).values())

    source_precision = source_overlap / total if total else 0
    target_precision = target_overlap / total if total else 0
    target_recall = target_overlap / sum(t.values()) if sum(t.values()) else 0

    target_f1 = (
        2 * target_precision * target_recall /
        (target_precision + target_recall)
        if target_precision + target_recall else 0
    )

    return source_precision, target_f1


metrics = [
    calculate_metrics(p, s, t)
    for p, s, t in zip(
        val_predictions,
        val_run["Abstract"],
        val_run["Highlights"]
    )
]

benchmark = {
    "model": "pegasus-pubmed",
    "model_id": MODEL_ID,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "evaluation_samples": len(val_predictions),
    "overall_quality_rougeL_f1": float(np.mean(rouge_scores)),
    "f1_score_target": float(np.mean([m[1] for m in metrics])),
    "source_grounding_precision": float(np.mean([m[0] for m in metrics])),
    "hallucination_risk_pct": float(
        1 - np.mean([m[0] for m in metrics])
    ),
    "wall_clock_latency_sec": float(np.mean(val_latencies)),
    "total_wall_clock_sec": float(np.sum(val_latencies)),
    "throughput_tokens_per_sec": float(
        np.sum(val_tokens) / max(np.sum(val_latencies), 1e-6)
    )
}

# 4. Save benchmark
benchmark_path = os.path.join(
    RESULTS_DIR, "benchmark_pegasus-pubmed.json"
)

with open(benchmark_path, "w") as f:
    json.dump(benchmark, f, indent=4)

print("\n========== BENCHMARK RESULTS ==========")

for key, value in benchmark.items():
    print(f"{key}: {value}")

# 5. Generate masked test predictions
print("\nGenerating test predictions...")

test_predictions, _, _ = generate_predictions(
    test_run["Abstract"].astype(str).tolist()
)

test_run["Highlights"] = test_predictions

output_path = os.path.join(
    RESULTS_DIR,
    "MixSub-SciHigh_test_predictions_pegasus-pubmed.csv"
)

test_run.to_csv(output_path, index=False)

# 6. Save validation predictions too
val_output = val_run.copy()
val_output["Predicted_Highlights"] = val_predictions

val_output.to_csv(
    os.path.join(RESULTS_DIR, "pegasus_validation_predictions.csv"),
    index=False
)

print("\n======================================")
print("SUCCESS!")
print("Benchmark:", benchmark_path)
print("Test predictions:", output_path)
print("======================================")